# Vietnamese Budget Forcing — Kaggle Runner

**Research question:** Does Test-Time Scaling via Budget Forcing transfer to Vietnamese language reasoning?

**Before running:**
1. Settings → Accelerator → **GPU T4 x1** (or P100)
2. Settings → Internet → **On**
3. Add your HuggingFace token: Settings → Secrets → Add `HF_TOKEN`

**Run order:** Cell 1 (GPU check) → Cell 2 (clone) → Cell 3 (install) → Cell 4 (HF auth) → Cell 5 (validate benchmarks) → Cell 6 (VRAM check) → Cell 7 (run sweep) → Cell 8 (aggregate) → Cell 9 (copy outputs)

**Scope:** BF-only, n_wait ∈ {0,1,2}. No retrieval.

In [1]:
# Cell 1 — Check GPU and environment
import subprocess, os, sys

print('=== GPU ===')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv'], check=False)

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

WORK_DIR = '/kaggle/working'
print(f'\nWorking dir: {WORK_DIR}')
print('Disk free: ', end='')
subprocess.run(['df', '-h', WORK_DIR])

=== GPU ===
name, memory.total [MiB], memory.free [MiB]
Tesla T4, 15360 MiB, 14912 MiB
Tesla T4, 15360 MiB, 14912 MiB

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB

Working dir: /kaggle/working
Disk free: Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  132K   20G   1% /kaggle/working


CompletedProcess(args=['df', '-h', '/kaggle/working'], returncode=0)

In [2]:
import os
import shutil

REPO_DIR = '/kaggle/working/s1_budget_forcing'
INPUT_PATH = '/kaggle/input/s1-budget-forcing'

# 1. If it's already there (e.g., interactive re-runs), skip setup
if os.path.exists(REPO_DIR):
    print(f'Repo already exists at {REPO_DIR}')

# 2. Try Option B first: If a Kaggle Dataset is attached, copy it locally
elif os.path.exists(INPUT_PATH):
    shutil.copytree(INPUT_PATH, REPO_DIR)
    print(f'Copied from Kaggle dataset to {REPO_DIR}')

# 3. Try Option A: Fallback to GitHub cloning if no dataset is attached
else:
    print(f'Dataset not found. Cloning from GitHub to {REPO_DIR}...')
    # Using subprocess to handle cloning reliably in background runs
    import subprocess
    result = subprocess.run(['git', 'clone', 'https://github.com/ajvan2808/s1_budget_forcing.git', REPO_DIR], capture_output=True, text=True)
    
    if result.returncode == 0:
        print("GitHub clone successful!")
    else:
        print(result.stderr)
        raise FileNotFoundError('Failed to clone from GitHub and no Kaggle Dataset was found.')

# Always move to the directory at the end
os.chdir(REPO_DIR)
print(f'CWD set to: {os.getcwd()}')

Dataset not found. Cloning from GitHub to /kaggle/working/s1_budget_forcing...
GitHub clone successful!
CWD set to: /kaggle/working/s1_budget_forcing


In [3]:
# Cell 3 — Install dependencies
# Kaggle has torch/transformers pre-installed; only install what's missing.
# No RAG dependencies needed — BF-only study.

!git checkout origin/feature

!uv sync

Note: switching to 'origin/feature'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 651bd01 Merge remote-tracking branch 'origin/feature' into feature
Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment at: .venv
Resolved 180 packages in 41ms
⠙ Preparing packages... (0/168)
⠙ Preparing packages... (0/168)
⠙ Preparing packages... (0/168)
pygments             ------------------------------     0 B/1.17 MiB
⠙ Preparing packages... (0/168)
pygments             -------------------

In [4]:
# Cell 4 — HuggingFace authentication
# Required for r1-distill-7B and Vietnamese models.
# Store your token in Kaggle Secrets as HF_TOKEN — never hardcode it here.
from kaggle_secrets import UserSecretsClient
import os

try:
    secret = UserSecretsClient()
    hf_token = secret.get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print('HuggingFace login successful.')
except Exception as e:
    print(f'[WARN] HF auth skipped: {e}')
    print('Public models (Qwen2.5, SeaLLM) will still work without a token.')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HuggingFace login successful.


In [5]:
!uv pip install bitsandbytes>=0.46.1  

Using Python 3.12.13 environment at: /usr
Resolved 31 packages in 5.40s
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
⠹ Preparing packages... (0/1)
Prepared 1 package in 315ms
Uninstalled 1 package in 19ms
Installed 2 packages in 8ms
 + bitsandbytes==0.49.2
 - cuda-bindings==13.2.0
 + cuda-bindings==12.9.4


In [6]:
# Cell 5 — Validate Vietnamese benchmarks
import sys, os
sys.path.insert(0, '/kaggle/working/s1_budget_forcing/experiments')
os.chdir('/kaggle/working/s1_budget_forcing')

!python experiments/data/download_vi_benchmarks.py --n_samples 2


Benchmark: vi_gsm8k
  Multilingual GSM8K — Vietnamese split (250 problems)
  Source: hllj/vi_gsm8k
README.md: 1.85kB [00:00, 4.90MB/s]
vi_train.json: 5.50MB [00:00, 71.8MB/s]
vi_test.json: 1.01MB [00:00, 126MB/s]
Generating test split: 100%|██████| 1319/1319 [00:00<00:00, 50162.18 examples/s]
[OK]  Loaded. Size: 1319 samples
      Fields: ['index', 'explanation', 'question', 'answer']
[OK]  All expected fields present: ['question', 'answer', 'index', 'explanation']

--- First 2 samples ---

Sample 1:
  index: 0
  explanation: 'Janet bán 9 quả trứng vịt mỗi ngày.\nCô ấy kiếm được 18 đô la mỗi ngày tại chợ nông sản.'
  question: 'Vịt của Janet đẻ được 16 quả trứng mỗi ngày. Cô ấy ăn 3 quả vào bữa sáng và nướng bánh muffin cho bạ...'
  answer: '18'

Sample 2:
  index: 1
  explanation: 'Cần 2/2=1 cuộn sợi màu trắng.\nVì vậy tổng số lượng vải là 2+1=3 cuộn vải.'
  question: 'Một chiếc áo choàng cần 2 cuộn sợi màu xanh và một nửa số đó là sợi màu trắng. Tổng cộng cần bao nhi...'
  answer: '

In [7]:
# Cell 6 — VRAM check
# Verify each model fits in T4 VRAM (15 GB) under 4-bit quantization.
# r1-distill-7B and Vi-7B models: ~3.5 GB each at 4-bit.
# Run this before the sweep to catch OOM before wasting GPU time.
import sys, os
sys.path.insert(0, '/kaggle/working/s1_budget_forcing/experiments')
os.chdir('/kaggle/working/s1_budget_forcing')

from models.model_loader import SUPPORTED_MODELS, estimate_vram_gb
import torch

VI_MODELS = ['qwen2.5-3B', 'r1-distill-7B', 'vinallama-7b', 'vistral-7b', 'seallm-7b', 'greenmind-14b-r1']

if torch.cuda.is_available():
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU VRAM: {total_vram:.1f} GB')
else:
    total_vram = 0
    print('[WARN] No GPU detected — models will run on CPU (very slow)')

print()
print(f'{"Model":<20} {"HF ID":<45} {"VRAM 4-bit":>10} {"Fits T4":>8}')
print('-' * 90)
for name in VI_MODELS:
    hf_id = SUPPORTED_MODELS.get(name, 'NOT IN REGISTRY')
    vram = estimate_vram_gb(name, load_in_4bit=True)
    fits = '✓' if vram < total_vram * 0.9 else '✗ OOM risk'
    print(f'{name:<20} {hf_id:<45} {vram:>8.1f} GB {fits:>8}')

print()
print('Note: Vinallama/Vistral use </s> as EoT token (LLaMA-2/Mistral base).')
print('      If thinking_tokens stays 0 after n_wait>0, check EoT suppression in decoding.py.')

GPU VRAM: 15.6 GB

Model                HF ID                                         VRAM 4-bit  Fits T4
------------------------------------------------------------------------------------------
qwen2.5-3B           Qwen/Qwen2.5-3B-Instruct                           1.5 GB        ✓
r1-distill-7B        deepseek-ai/DeepSeek-R1-Distill-Qwen-7B            3.5 GB        ✓
vinallama-7b         vilm/vinallama-7b-chat                             3.5 GB        ✓
vistral-7b           Viet-Mistral/Vistral-7B-Chat                       3.5 GB        ✓
seallm-7b            SeaLLMs/SeaLLMs-v3-7B-Chat                         3.5 GB        ✓
greenmind-14b-r1     GreenNode/GreenMind-Medium-14B-R1                  7.0 GB        ✓

Note: Vinallama/Vistral use </s> as EoT token (LLaMA-2/Mistral base).
      If thinking_tokens stays 0 after n_wait>0, check EoT suppression in decoding.py.


In [8]:
# Cell 7 — Run experiments
#
# Set PRESET below, then Run All from this cell.
# Start with SMOKE to confirm pipeline, then scale up.
#
# SMOKE        1 model × 1 benchmark × 3 n_wait × 5 samples  → ~5 min
# SMALL_MATRIX 2 models × 2 benchmarks × 3 n_wait × 20 samples → ~30-45 min
# FULL         5 models × 2 benchmarks × 3 n_wait × 100 samples → ~3-5 hr

import os, subprocess
os.chdir('/kaggle/working/s1_budget_forcing')

# ── Choose a preset ──────────────────────────────────────────────────────────
PRESET = 'FULL_GREENMIND'   # 'SMOKE' | 'SMALL_MATRIX' | 'FULL'

PRESETS = {
    'SMOKE': dict(
        MODELS      = 'qwen2.5-3B',
        BENCHMARKS  = 'vi_gsm8k',
        N_WAIT_LIST = '0 1 2',
        N_SAMPLES   = '5',
        EXTRA_ARGS  = '--max_tokens 512',   # 4-bit ON by default on Kaggle GPU
    ),
    'SMALL_MATRIX': dict(
        MODELS      = 'r1-distill-7B',
        BENCHMARKS  = 'vimmlu',
        N_WAIT_LIST = '0 1 2',
        N_SAMPLES   = '20',
        EXTRA_ARGS  = '',
    ),
    'FULL_REASONING': dict(
        MODELS      = 'r1-distill-7B',
        BENCHMARKS  = 'vi_gsm8k vimmlu vnhsge',
        N_WAIT_LIST = '0 1 2',
        N_SAMPLES   = '100',
        EXTRA_ARGS  = '',
    ),
    'FULL_GREENMIND': dict(
        MODELS      = 'greenmind-14b-r1',
        BENCHMARKS  = 'vi_gsm8k vimmlu vnhsge',
        N_WAIT_LIST = '0 1 2',
        N_SAMPLES   = '100',
        EXTRA_ARGS  = '',
    ),
    'FULL_VI': dict(
        MODELS      = 'vinallama-7b vistral-7b seallm-7b',
        BENCHMARKS  = 'vi_gsm8k vimmlu vnhsge',
        N_WAIT_LIST = '0 1 2',
        N_SAMPLES   = '100',
        EXTRA_ARGS  = '',
    ),
}

cfg = PRESETS[PRESET]
print(f'Running preset: {PRESET}')
for k, v in cfg.items():
    print(f'  {k}={v!r}')

env = os.environ.copy()
env.update({
    'MODELS':      cfg['MODELS'],
    'BENCHMARKS':  cfg['BENCHMARKS'],
    'N_WAIT_LIST': cfg['N_WAIT_LIST'],
    'N_SAMPLES':   cfg['N_SAMPLES'],
    'OUTPUT_DIR':  'experiments/results',
    'EXTRA_ARGS':  cfg['EXTRA_ARGS'],
})

result = subprocess.run(
    ['bash', 'experiments/scripts/run_vi_bf.sh'],
    env=env,
    cwd='/kaggle/working/s1_budget_forcing',
)
print(f'\nExit code: {result.returncode}')

Running preset: FULL_GREENMIND
  MODELS='greenmind-14b-r1'
  BENCHMARKS='vi_gsm8k vimmlu vnhsge'
  N_WAIT_LIST='0 1 2'
  N_SAMPLES='100'
  EXTRA_ARGS=''
Vietnamese Budget Forcing Sweep (BF-only)
  Models:     greenmind-14b-r1
  Benchmarks: vi_gsm8k vimmlu vnhsge
  n_wait:     0 1 2
  n_samples:  100
  output_dir: experiments/results
  seed:       42

----------------------------------------------------------------------
  Model: greenmind-14b-r1  |  Benchmark: vi_gsm8k
----------------------------------------------------------------------
  Running python experiments/evaluation/run_eval_vi.py --model greenmind-14b-r1 --benchmark vi_gsm8k ...

[run_eval_vi] Started at 20260621_154600
  model=greenmind-14b-r1, benchmark=vi_gsm8k
  n_wait_list=[0, 1, 2], n_samples=100
  trigger='Chờ một chút', load_in_4bit=True, max_new_tokens=2048
[run_eval_vi] Loading benchmark: hllj/vi_gsm8k
[run_eval_vi] Sampled 100 questions (seed=42)
[run_eval_vi] Loading model: greenmind-14b-r1
Loading: GreenNode/G

Loading weights:  75%|███████▌  | 436/579 [01:56<00:38,  3.74it/s, Materializing param=model.layers.36.mlp.down_proj.weight]
Traceback (most recent call last):
  File "/kaggle/working/s1_budget_forcing/experiments/evaluation/run_eval_vi.py", line 420, in <module>
    run_evaluation_vi(
  File "/kaggle/working/s1_budget_forcing/experiments/evaluation/run_eval_vi.py", line 285, in run_evaluation_vi
    model, tokenizer = load_model_and_tokenizer(
                       ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/s1_budget_forcing/experiments/models/model_loader.py", line 78, in load_model_and_tokenizer
    model = AutoModelForCausalLM.from_pretrained(
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/auto_factory.py", line 372, in from_pretrained
    return model_class.from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py", line 

[FAIL] Cannot load model greenmind-14b-r1: CUDA out of memory. Tried to allocate 136.00 MiB. GPU 1 has a total capacity of 14.56 GiB of which 42.81 MiB is free. Including non-PyTorch memory, this process has 14.52 GiB memory in use. Of the allocated memory 14.39 GiB is allocated by PyTorch, and 10.07 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

  [FAIL] greenmind-14b-r1 × vi_gsm8k exited with code 1

----------------------------------------------------------------------
  Model: greenmind-14b-r1  |  Benchmark: vimmlu
----------------------------------------------------------------------
  Running python experiments/evaluation/run_eval_vi.py --model greenmind-14b-r1 --benchmark vimmlu ...

[run_eval_vi] Started at 20260621_155031
  model=greenmind-14b-r

Loading weights:  75%|███████▌  | 436/579 [02:22<00:46,  3.07it/s, Materializing param=model.layers.36.mlp.down_proj.weight]
Traceback (most recent call last):
  File "/kaggle/working/s1_budget_forcing/experiments/evaluation/run_eval_vi.py", line 420, in <module>
    run_evaluation_vi(
  File "/kaggle/working/s1_budget_forcing/experiments/evaluation/run_eval_vi.py", line 285, in run_evaluation_vi
    model, tokenizer = load_model_and_tokenizer(
                       ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/s1_budget_forcing/experiments/models/model_loader.py", line 78, in load_model_and_tokenizer
    model = AutoModelForCausalLM.from_pretrained(
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/auto_factory.py", line 372, in from_pretrained
    return model_class.from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py", line 

[FAIL] Cannot load model greenmind-14b-r1: CUDA out of memory. Tried to allocate 136.00 MiB. GPU 1 has a total capacity of 14.56 GiB of which 50.81 MiB is free. Including non-PyTorch memory, this process has 14.51 GiB memory in use. Of the allocated memory 14.29 GiB is allocated by PyTorch, and 107.08 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

  [FAIL] greenmind-14b-r1 × vimmlu exited with code 1

----------------------------------------------------------------------
  Model: greenmind-14b-r1  |  Benchmark: vnhsge
----------------------------------------------------------------------
  Running python experiments/evaluation/run_eval_vi.py --model greenmind-14b-r1 --benchmark vnhsge ...

[run_eval_vi] Started at 20260621_155314
  model=greenmind-14b-r1

Loading weights:  75%|███████▌  | 436/579 [02:17<00:45,  3.17it/s, Materializing param=model.layers.36.mlp.down_proj.weight]
Traceback (most recent call last):
  File "/kaggle/working/s1_budget_forcing/experiments/evaluation/run_eval_vi.py", line 420, in <module>
    run_evaluation_vi(
  File "/kaggle/working/s1_budget_forcing/experiments/evaluation/run_eval_vi.py", line 285, in run_evaluation_vi
    model, tokenizer = load_model_and_tokenizer(
                       ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/s1_budget_forcing/experiments/models/model_loader.py", line 78, in load_model_and_tokenizer
    model = AutoModelForCausalLM.from_pretrained(
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/auto/auto_factory.py", line 372, in from_pretrained
    return model_class.from_pretrained(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py", line 

[FAIL] Cannot load model greenmind-14b-r1: CUDA out of memory. Tried to allocate 136.00 MiB. GPU 1 has a total capacity of 14.56 GiB of which 40.81 MiB is free. Including non-PyTorch memory, this process has 14.52 GiB memory in use. Of the allocated memory 14.38 GiB is allocated by PyTorch, and 20.51 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

  [FAIL] greenmind-14b-r1 × vnhsge exited with code 1

Sweep complete: 0/3 runs succeeded.

Aggregate results with:
  python experiments/results/summary_vi.py \
      --results_dir experiments/results

Exit code: 1


In [9]:
# Cell 8 — Aggregate results
import glob, os

results_root = '/kaggle/working/s1_budget_forcing/experiments/results'
run_dirs = sorted(glob.glob(f'{results_root}/vi_*/'))

if not run_dirs:
    print('No vi_* result directories found. Run Cell 7 first.')
else:
    latest = run_dirs[-1]
    print(f'Aggregating: {latest}')
    !python experiments/results/summary_vi.py --results_dir {latest}

    import pandas as pd
    csv_path = os.path.join(latest, 'summary_vi.csv')
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        display_cols = ['model', 'benchmark', 'n_wait', 'n_samples',
                        'accuracy', 'scaling', 'performance',
                        'avg_thinking_tokens', 'extraction_failures']
        print(df[display_cols].to_string(index=False))
    else:
        print(f'CSV not found at {csv_path}')

Aggregating: /kaggle/working/s1_budget_forcing/experiments/results/vi_20260621_155314/
[WARN] No JSON files found in /kaggle/working/s1_budget_forcing/experiments/results/vi_20260621_155314
[ERROR] No result files found. Check --results_dir.
CSV not found at /kaggle/working/s1_budget_forcing/experiments/results/vi_20260621_155314/summary_vi.csv


In [10]:
# Cell 9 — Copy outputs to /kaggle/working for download
# Kaggle allows downloading files from /kaggle/working/.
import shutil, os, glob

OUT = '/kaggle/working/outputs'
os.makedirs(OUT, exist_ok=True)

for d in glob.glob('/kaggle/working/s1_budget_forcing/experiments/results/vi_*'):
    dest = os.path.join(OUT, os.path.basename(d))
    if not os.path.exists(dest):
        shutil.copytree(d, dest)
        print(f'Copied: {dest}')

print(f'\nAll outputs in: {OUT}')
!ls -lh {OUT}

Copied: /kaggle/working/outputs/vi_20260621_155031
Copied: /kaggle/working/outputs/vi_20260621_155314
Copied: /kaggle/working/outputs/vi_20260621_154600

All outputs in: /kaggle/working/outputs
total 12K
drwxr-xr-x 2 root root 4.0K Jun 21 08:46 vi_20260621_154600
drwxr-xr-x 2 root root 4.0K Jun 21 08:50 vi_20260621_155031
drwxr-xr-x 2 root root 4.0K Jun 21 08:53 vi_20260621_155314
